# Imports

In [1]:
import torch
import numpy as np

from sklearn.model_selection import train_test_split
from transformers import BertModel, BertConfig
from torch.utils.data import DataLoader

# Data handeling
from src.data.dataset import MLMDataset, ClassificationDataset, HieraricalClassificationDataset, OverlappingKmerHieraricalClassificationDataset
from src.data.data_tools import filter_taxonomy, fasta2pandas

# Vocab shit
from src.utils.vocab import Vocabulary, KmerVocabConstructor

# Preprocessing
from src.preprocessing.augmentation import SequenceModifier, IdentityStrategy, BaseStrategy
from src.preprocessing.tokenization import KmerStrategy
from src.preprocessing.padding import PEndStrategy
from src.preprocessing.truncation import TEndStrategy
from src.preprocessing.preprocessor import Preprocessor

# Model things
from src.model.backbone import Bertax, ModularBertax, OverlappingKmerModularBertax
from src.model.encoders import LabelEncoder
from src.model.heads import MLMHead, SingleClassHead, HierarchicalClassificationHead

# Training things
from src.train.trainers import MLMtrainer, ClassificationTrainer, HierarchicalClassificationTrainer

c:\Users\user\anaconda3\Lib\site-packages\transformers\utils\generic.py:260: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  torch.utils._pytree._register_pytree_node(


# Config

In [2]:
CONFIG = {
    "FILE_PATH": "src/data/raw.fasta",
    "SAVE_PATH": "pretrained_model.pt",
    "n_test": 500,
    "modification_probability": 0.05,
    "alphabet": ["A", "C", "G", "T"],
    "k": 3,
    "optimal_length": 200,

    # Training parameters
    "num_epochs": 1,
    "masking_percentage": 0.05,
    "batch_size": 128,
    "small_set": True,

    # Model configuration
    "num_layers": 10,
    "num_attention_heads": 4,
    "hidden_size": 256,
    "intermediate_size": 1024,  # 4 * hidden_size
    "dropout_rate": 0.05,
    "num_classes": 19,
    "mlm_dropout_rate": 0.1,

    #Classification 
    "target_labels": ["phylum", "class", "order"]
}

In [3]:
# Set up vocabulary
constructor = KmerVocabConstructor(k=CONFIG["k"], alphabet=CONFIG["alphabet"])
vocab = Vocabulary()
vocab.build_from_constructor(constructor, data=[])
vocab_path = "vocab.json"
vocab.save(vocab_path)

# Set up preprocessors
sequence_modifier = SequenceModifier(alphabet=CONFIG["alphabet"])
augmentation_strategy_train = BaseStrategy(
    modifier=sequence_modifier,
    alphabet=CONFIG["alphabet"],
    modification_probability=CONFIG["modification_probability"]
)

augmentation_strategy_val = IdentityStrategy(
    modifier=sequence_modifier,
    alphabet=CONFIG["alphabet"],
    modification_probability=0
)

tokenization_strategy = KmerStrategy(
    k=CONFIG["k"],
    padding_alphabet=CONFIG["alphabet"]
)

padding_strategy = PEndStrategy(
    optimal_length=CONFIG["optimal_length"]
)

truncation_strategy = TEndStrategy(
    optimal_length=CONFIG["optimal_length"]
)

preprocessor_train = Preprocessor(
    augmentation_strategy=augmentation_strategy_train,
    tokenization_strategy=tokenization_strategy,
    padding_strategy=padding_strategy,
    truncation_strategy=truncation_strategy,
    vocab=vocab,
)

preprocessor_val = Preprocessor(
    augmentation_strategy=augmentation_strategy_val,
    tokenization_strategy=tokenization_strategy,
    padding_strategy=padding_strategy,
    truncation_strategy=truncation_strategy,
    vocab=vocab,
)


In [4]:
##################################################################
## Data preparation ##############################################
##################################################################

all_data = fasta2pandas(CONFIG["FILE_PATH"])

#tmp: to use a smaller set for test if code compiles - nothing to be used in production
if CONFIG["small_set"]:
    all_data = all_data[:CONFIG["n_test"]]

target_labels = CONFIG["target_labels"]

# filter data for finetuning
filtered_data = filter_taxonomy(
    df = all_data,
    startAt =target_labels[0],
    endAt = target_labels[-1],
    phylumCertainty = True
    )

class_sizes = [len(list(set(filtered_data[target_label]))) for target_label in target_labels]

label_encoders = {}
for target_label in target_labels:
    label_encoders[target_label] = LabelEncoder(list(set(filtered_data[target_label])))
    
# Pretraining datasplit based on unsplit data: 
pretrain_df, preval_df = train_test_split(
    all_data,
    test_size = 0.1,
    random_state = 42
)

# Finetune datasplit based on filtered data:
finetrain_data, fineval_data = train_test_split(
    filtered_data,
    test_size = 0.1,
    random_state = 69
    )

print(f"Number of pre-training sequences: {len(pretrain_df)}")
print(f"Number of validation sequences: {len(preval_df)}")


Applying filters...
Filtering complete.

Handling DNA ambiguity codes...
Processing row 0...
Processing complete.
Number of pre-training sequences: 450
Number of validation sequences: 50


In [5]:
##################################################################
## Setup Datasets ################################################
##################################################################

# Datasets
pretrain_dataset = MLMDataset(
    df = pretrain_df,
    preprocessor = preprocessor_train,
    masking_percentage = CONFIG["masking_percentage"]
)

preval_dataset = MLMDataset(
    df = preval_df,
    preprocessor = preprocessor_val,
    masking_percentage = CONFIG["masking_percentage"]
)

# Finetuning datasets
finetrain_dataset = OverlappingKmerHieraricalClassificationDataset(
    df = finetrain_data,
    k = CONFIG["k"],
    augmenter = augmentation_strategy_train,
    preprocessor = preprocessor_val,
    label_encoders = label_encoders,
    )

fineval_dataset = OverlappingKmerHieraricalClassificationDataset(
    df = fineval_data,
    k = CONFIG["k"],
    augmenter = augmentation_strategy_val,
    preprocessor = preprocessor_val,
    label_encoders = label_encoders,
    )



In [6]:
##################################################################
## Setup Dataloader ##############################################
##################################################################

pretrain_loader = DataLoader(
    dataset = pretrain_dataset,
    batch_size = CONFIG["batch_size"],
    shuffle = True
)

preval_loader = DataLoader(
    dataset = preval_dataset,
    batch_size = CONFIG["batch_size"],
    shuffle = True
)

#data loaders
finetrain_loader = DataLoader(
    dataset=finetrain_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True
    )

fineval_loader = DataLoader(
    dataset=fineval_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True
    )

In [7]:
encoder_config = BertConfig(
    vocab_size = len(vocab),
    hidden_size = CONFIG["hidden_size"],
    num_hidden_layers = CONFIG["num_layers"],
    num_attention_heads = CONFIG["num_attention_heads"],
    intermediate_size = CONFIG["intermediate_size"],
    max_position_embeddings = CONFIG["optimal_length"] + 2,
    hidden_dropout_prob = CONFIG["dropout_rate"],
    attention_probs_dropout_prob = CONFIG["dropout_rate"]
)

encoder = BertModel(encoder_config)

mlm_head = MLMHead(
    in_features = CONFIG["hidden_size"],
    hidden_layer_size = CONFIG["hidden_size"] // 2,
    out_features = len(vocab),
    dropout_rate = CONFIG["mlm_dropout_rate"]
)


classification_head = HierarchicalClassificationHead(
    in_features=CONFIG["hidden_size"],
    class_sizes= class_sizes,
    dropout_rate=CONFIG["dropout_rate"]
    )

model = OverlappingKmerModularBertax(
    encoder = encoder,
    mlm_head = mlm_head,
    classification_head = classification_head
)

In [12]:
print(model)
print(type(model))

OverlappingKmerModularBertax(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(69, 256, padding_idx=0)
      (position_embeddings): Embedding(202, 256)
      (token_type_embeddings): Embedding(2, 256)
      (LayerNorm): LayerNorm((256,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.05, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-9): 10 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=256, out_features=256, bias=True)
              (key): Linear(in_features=256, out_features=256, bias=True)
              (value): Linear(in_features=256, out_features=256, bias=True)
              (dropout): Dropout(p=0.05, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=256, out_features=256, bias=True)
              (LayerNorm): LayerNorm((256,), eps=1e-12, el

# Testing out that it works

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
item = finetrain_dataset[0]
input_ids = item["input_ids"].to(device)          # shape [k, seq_len]
attention_mask = item["attention_mask"].to(device)

model = OverlappingKmerModularBertax(encoder, mlm_head, classification_head)

# Pretrain
model.preTrainMode()
mlm_logits = model(input_ids, attention_mask)  
# => shape [1, seq_len, vocab_size], or something similar

# Classify
model.classifyMode()
hier_logits = model(input_ids, attention_mask)
# => typically a list of classification outputs, e.g. [ [1, num_classes_lvl1], [1, num_classes_lvl2], ... ]


# Training